In [1]:
# pipeline/run_ingestion.py
import csv
import asyncio
from pathlib import Path

CUAD_ROOT = Path("../CUAD_v1")
CSV_PATH  = CUAD_ROOT / "master_clauses.csv"

# All clause columns (without -Answer suffix)
RISK_CLAUSE_COLS = [
    "Termination For Convenience",
    "Cap On Liability",
    "Uncapped Liability",
    "Anti-Assignment",
    "Change Of Control",
    "Ip Ownership Assignment",
    "Non-Compete",
    "Exclusivity",
    "Liquidated Damages",
    "Audit Rights",
    "Insurance",
    "Minimum Commitment",
    "Governing Law",
    "Renewal Term",
    "Notice Period To Terminate Renewal",
]

def load_contracts_from_csv(limit=None):
    """
    Reads master_clauses.csv and returns list of contracts
    with their PDF path and ground-truth clauses.
    """
    contracts = []
    
    with open(CSV_PATH, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        
        for row in reader:
            filename = row['Filename']
            
            # Find the PDF — search all Part folders
            pdf_path = find_pdf(filename)
            if not pdf_path:
                print(f"  WARNING: PDF not found for {filename}")
                continue
            
            # Extract contract type from filename
            # e.g. "...Affiliate Agreement.pdf" → "Affiliate Agreement"
            contract_type = filename.replace('.pdf', '').split('_')[-1].strip()
            
            # Build ground truth clauses from CSV
            ground_truth_clauses = []
            for col in RISK_CLAUSE_COLS:
                answer_col = col + '-Answer'
                answer = row.get(answer_col, '').strip()
                text   = row.get(col, '').strip()
                
                # Only include clauses that ARE present
                if answer == 'Yes' and text and text != '[]':
                    # Clean the text — CSV stores as Python list string
                    clean_text = text.strip("[]'\"").replace("', '", " | ")
                    ground_truth_clauses.append({
                        "clause_type":    col,
                        "text":           clean_text[:1000],
                        "answer":         answer,
                        "is_cuad_labeled": True,
                    })
            
            contracts.append({
                "filename":              filename,
                "local_pdf_path":        str(pdf_path),
                "contract_type":         contract_type,
                "source":                "cuad",
                "ground_truth_clauses":  ground_truth_clauses,
                "num_risk_clauses":      len(ground_truth_clauses),
            })
            
            if limit and len(contracts) >= limit:
                break
    
    return contracts

def find_pdf(filename: str) -> Path | None:
    """Search Part_I, Part_II, Part_III for the PDF."""
    for part in ["Part_I", "Part_II", "Part_III"]:
        part_dir = CUAD_ROOT / "full_contract_pdf" / part
        if not part_dir.exists():
            continue
        # Search all subdirectories
        for pdf in part_dir.rglob(filename):
            return pdf
    return None

def print_summary(contracts):
    print(f"\nTotal contracts found: {len(contracts)}")
    
    # By contract type
    from collections import Counter
    types = Counter(c['contract_type'] for c in contracts)
    print(f"\nTop contract types:")
    for ctype, count in types.most_common(10):
        print(f"  {ctype}: {count}")
    
    # Risk distribution
    print(f"\nRisk clause distribution:")
    from collections import defaultdict
    clause_counts = defaultdict(int)
    for c in contracts:
        for clause in c['ground_truth_clauses']:
            clause_counts[clause['clause_type']] += 1
    
    for clause, count in sorted(clause_counts.items(), 
                                  key=lambda x: -x[1]):
        pct = count / len(contracts) * 100
        print(f"  {clause:45} {count:3} ({pct:.0f}%)")
    
    # Contracts with most risks
    print(f"\nTop 5 highest-risk contracts:")
    top = sorted(contracts, key=lambda x: -x['num_risk_clauses'])[:5]
    for c in top:
        print(f"  {c['filename'][:60]}")
        print(f"    Risk clauses: {c['num_risk_clauses']}")
        for clause in c['ground_truth_clauses']:
            print(f"      ✓ {clause['clause_type']}")

if __name__ == "__main__":
    print("Loading contracts from CSV...")
    contracts = load_contracts_from_csv(limit=None)
    print_summary(contracts)
    
    # Save for the ingestion pipeline
    import json
    output_path = Path("eval/contracts_manifest.json")
    output_path.parent.mkdir(exist_ok=True)
    output_path.write_text(json.dumps(contracts[:10], indent=2))
    print(f"\nSaved first 10 to eval/contracts_manifest.json")

Loading contracts from CSV...

Total contracts found: 499

Top contract types:
  Development Agreement: 20
  Co-Branding Agreement: 17
  Content License Agreement: 16
  Endorsement Agreement: 13
  Distributor Agreement: 11
  Affiliate Agreement: 8
  Supply Agreement: 6
  Reseller Agreement: 6
  Service Agreement: 5
  Trademark License Agreement: 5

Risk clause distribution:
  Anti-Assignment                               366 (73%)
  Cap On Liability                              267 (54%)
  Audit Rights                                  208 (42%)
  Termination For Convenience                   179 (36%)
  Exclusivity                                   174 (35%)
  Minimum Commitment                            164 (33%)
  Insurance                                     159 (32%)
  Ip Ownership Assignment                       121 (24%)
  Change Of Control                             115 (23%)
  Non-Compete                                   113 (23%)
  Uncapped Liability                       